> Repository edition: run from the repository root using an environment with the dependencies in `requirements.txt`. Machine-specific environment paths have been replaced. Historical outputs are retained except environment checks; they are not evidence of a fresh rerun. See `docs/REPRODUCIBILITY.md`.


In [1]:
from pathlib import Path
import importlib.util
import sys
import numpy as np
import pandas as pd

# Check the Python environment and working directory
print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())

# Check the initial data and cumulative feedback files
print("Initial data folder found:", Path("initial_data").is_dir())
print("Cumulative inputs found:", Path("week_3/inputs.txt").is_file())
print("Cumulative outputs found:", Path("week_3/outputs.txt").is_file())

# Check available modelling libraries without importing them yet
for package in ["sklearn", "matplotlib", "torch", "tensorflow"]:
    available = importlib.util.find_spec(package) is not None
    print(f"{package} available:", available)

Python executable: c:\Users\AG\BeatrizGIT\Master\.venv\Scripts\python.exe
Working directory: c:\Users\AG\BeatrizGIT\Master\CapstoneProject
Initial data folder found: True
Cumulative inputs found: True
Cumulative outputs found: True
sklearn available: True
matplotlib available: True
torch available: True
tensorflow available: True


In [2]:
import ast
import re

# Read cumulative feedback containing rounds 1, 2 and 3
inputs_text = Path("week_3/inputs.txt").read_text(
    encoding="utf-8-sig"
)
outputs_text = Path("week_3/outputs.txt").read_text(
    encoding="utf-8-sig"
)

# Convert NumPy array representations into plain lists
inputs_text = re.sub(
    r"array\((\[[^\[\]]*\])\)",
    r"\1",
    inputs_text
)

# Safely read each round, including lists spanning multiple lines
def read_rounds(text):
    tree = ast.parse(text.strip(), mode="exec")
    rounds = []

    for node in tree.body:
        if not isinstance(node, ast.Expr):
            raise ValueError("Unexpected format in the feedback file.")
        rounds.append(ast.literal_eval(node.value))

    return rounds


input_rounds = read_rounds(inputs_text)
output_rounds = read_rounds(outputs_text)

# Validate the number of rounds and functions
assert len(input_rounds) == len(output_rounds) == 3, (
    "Expected exactly three rounds in each cumulative file."
)

for round_inputs, round_outputs in zip(input_rounds, output_rounds):
    assert len(round_inputs) == len(round_outputs) == 8, (
        "Each round must contain results for eight functions."
    )

# Rebuild from the initial files to avoid duplication when rerunning
initial_data = {}
updated_data = {}

for function_number in range(1, 9):
    folder = Path("initial_data") / f"function_{function_number}"

    initial_X = np.load(folder / "initial_inputs.npy")
    initial_y = np.load(folder / "initial_outputs.npy").reshape(-1)

    new_X = np.asarray(
        [r[function_number - 1] for r in input_rounds],
        dtype=float
    )
    new_y = np.asarray(
        [r[function_number - 1] for r in output_rounds],
        dtype=float
    )

    assert initial_X.ndim == 2
    assert initial_X.shape[0] == initial_y.size
    assert new_X.shape == (3, initial_X.shape[1])
    assert new_y.shape == (3,)

    X = np.vstack([initial_X, new_X])
    y = np.concatenate([initial_y, new_y])

    assert np.isfinite(X).all() and np.isfinite(y).all()

    initial_data[function_number] = {
        "X": initial_X.copy(),
        "y": initial_y.copy()
    }
    updated_data[function_number] = {"X": X, "y": y}

    print(
        f"Function {function_number}: "
        f"{len(initial_y)} → {len(y)} observations | "
        f"{X.shape[1]} variables"
    )

Function 1: 10 → 13 observations | 2 variables
Function 2: 10 → 13 observations | 2 variables
Function 3: 15 → 18 observations | 3 variables
Function 4: 30 → 33 observations | 4 variables
Function 5: 20 → 23 observations | 4 variables
Function 6: 20 → 23 observations | 5 variables
Function 7: 30 → 33 observations | 6 variables
Function 8: 40 → 43 observations | 8 variables


**Data loading and validation**

The initial observations were combined with the results from all three completed rounds. Functions 1–8 now contain 13, 13, 18, 33, 23, 23, 33 and 43 observations, respectively, with their original input dimensions preserved.

The checks confirmed matching input–output dimensions and finite numerical values. Each feedback round was incorporated once. Rerunning this cell rebuilds the data sets from the original files rather than appending the same observations again.


In [4]:
from IPython.display import display

# Summarise progress across the three completed rounds
progress_rows = []

for function_number in range(1, 9):
    initial_y = initial_data[function_number]["y"]
    initial_best = float(np.max(initial_y))

    round_outputs = [
        float(r[function_number - 1])
        for r in output_rounds
    ]

    best_after_round_1 = max(initial_best, round_outputs[0])
    best_after_round_2 = max(best_after_round_1, round_outputs[1])
    best_after_round_3 = max(best_after_round_2, round_outputs[2])

    progress_rows.append({
        "Function": function_number,
        "Initial best": initial_best,
        "Best after round 1": best_after_round_1,
        "Best after round 2": best_after_round_2,
        "Round 3 output": round_outputs[2],
        "Round 3 improved": round_outputs[2] > best_after_round_2,
        "Round 3 improvement": best_after_round_3 - best_after_round_2,
        "Current best": best_after_round_3
    })

progress_summary = pd.DataFrame(progress_rows).set_index("Function")

# Use scientific notation to preserve very small output values
numeric_columns = [
    "Initial best",
    "Best after round 1",
    "Best after round 2",
    "Round 3 output",
    "Round 3 improvement",
    "Current best"
]

display(
    progress_summary.style.format({
        column: "{:.6e}" for column in numeric_columns
    })
)

improved_functions = progress_summary.index[
    progress_summary["Round 3 improved"]
].tolist()

print("Functions improved in round 3:", improved_functions)
print("Number of functions improved:", len(improved_functions))

,Initial best,Best after round 1,Best after round 2,Round 3 output,Round 3 improved,Round 3 improvement,Current best
Function,,,,,,,
1,7.710875e-16,7.710875e-16,7.710875e-16,1.210848e-51,False,0.000000e+00,7.710875e-16
2,6.112052e-01,6.582395e-01,6.582395e-01,3.826259e-01,False,0.000000e+00,6.582395e-01
3,-3.483531e-02,-3.483531e-02,-3.483531e-02,-1.646276e-01,False,0.000000e+00,-3.483531e-02
4,-4.025542e+00,1.948721e-01,1.948721e-01,-5.808035e-01,False,0.000000e+00,1.948721e-01
5,1.088860e+03,2.612732e+03,2.612732e+03,4.215408e+03,True,1.602677e+03,4.215408e+03
6,-7.142649e-01,-7.142649e-01,-6.174861e-01,-6.493585e-01,False,0.000000e+00,-6.174861e-01
7,1.364968e+00,2.097864e+00,2.097864e+00,1.774083e-01,False,0.000000e+00,2.097864e+00
8,9.598482e+00,9.802623e+00,9.820152e+00,9.850738e+00,True,3.058591e-02,9.850738e+00


Functions improved in round 3: [5, 8]
Number of functions improved: 2


**Progress after round 3**

Functions 5 and 8 improved their best observed outputs. Function 5 increased from approximately 2612.73 to 4215.41, while Function 8 increased from 9.82015 to 9.85074. The other six functions retained their previous best values.

These results show where the latest queries achieved improvements, but they do not establish how accurate the model predictions were. Comparing predicted and observed outputs will help identify where the surrogate model may need revision.


In [5]:
# Historical predictions from the selected round-3 GP models
# Values were recorded before receiving the actual portal outputs
round_3_predictions = {
    1: {"mean": -2.502833e-04, "std": 9.996244e-04},
    2: {"mean": 5.539196e-01, "std": 1.214022e-01},
    3: {"mean": -9.603975e-02, "std": 9.390945e-02},
    4: {"mean": -1.633536e-01, "std": 5.466365e-01},
    5: {"mean": 2.668558e+03, "std": 9.047195e+01},
    6: {"mean": -5.867368e-01, "std": 1.676386e-01},
    7: {"mean": 1.976501e+00, "std": 1.616568e-01},
    8: {"mean": 9.840428e+00, "std": 1.473795e-01}
}

prediction_rows = []

for function_number in range(1, 9):
    prediction = round_3_predictions[function_number]
    observed = float(output_rounds[2][function_number - 1])

    predicted_mean = prediction["mean"]
    predicted_std = prediction["std"]

    # Positive error means the model underestimated the output
    error = observed - predicted_mean

    prediction_rows.append({
        "Function": function_number,
        "Predicted mean": predicted_mean,
        "Predicted std": predicted_std,
        "Observed output": observed,
        "Observed minus predicted": error,
        "Absolute error / predicted std":
            abs(error) / predicted_std
    })

prediction_review = pd.DataFrame(
    prediction_rows
).set_index("Function")

display(
    prediction_review.style.format({
        "Predicted mean": "{:.6e}",
        "Predicted std": "{:.6e}",
        "Observed output": "{:.6e}",
        "Observed minus predicted": "{:.6e}",
        "Absolute error / predicted std": "{:.2f}"
    })
)

,Predicted mean,Predicted std,Observed output,Observed minus predicted,Absolute error / predicted std
Function,,,,,
1,-2.502833e-04,9.996244e-04,1.210848e-51,2.502833e-04,0.25
2,5.539196e-01,1.214022e-01,3.826259e-01,-1.712937e-01,1.41
3,-9.603975e-02,9.390945e-02,-1.646276e-01,-6.858782e-02,0.73
4,-1.633536e-01,5.466365e-01,-5.808035e-01,-4.174499e-01,0.76
5,2.668558e+03,9.047195e+01,4.215408e+03,1.546850e+03,17.10
6,-5.867368e-01,1.676386e-01,-6.493585e-01,-6.262168e-02,0.37
7,1.976501e+00,1.616568e-01,1.774083e-01,-1.799093e+00,11.13
8,9.840428e+00,1.473795e-01,9.850738e+00,1.030980e-02,0.07


**Review of prediction errors**

Functions 5 and 7 showed the largest discrepancies between predicted and observed outputs. Function 5 was underestimated by approximately 1546.85, whereas Function 7 was overestimated by approximately 1.80. These errors correspond to 17.10 and 11.13 predicted standard deviations, respectively, suggesting that the GP uncertainty estimates were too narrow at these particular query points.

Function 8's prediction was close to its observed output, with an error of approximately 0.0103. However, one result per function is insufficient to establish overall uncertainty calibration. These findings motivate comparing alternative models, especially for Functions 5 and 7, rather than assuming that greater model complexity will improve performance.


In [7]:
# Prepare binary labels using the same thresholds as in Module 14
classification_data = {}
classification_rows = []

for function_number in range(1, 9):
    X = updated_data[function_number]["X"]
    y = updated_data[function_number]["y"]

    # Keep the threshold fixed using only the initial observations
    threshold = float(
        np.median(initial_data[function_number]["y"])
    )

    # Class 1: above the initial median
    # Class 0: at or below the initial median
    labels = (y > threshold).astype(int)

    class_counts = np.bincount(labels, minlength=2)

    # Ensure each class can be represented in three validation folds
    assert class_counts.min() >= 3, (
        f"Function {function_number}: "
        "not enough observations per class for three-fold validation."
    )

    classification_data[function_number] = {
        "X": X.copy(),
        "y": y.copy(),
        "labels": labels,
        "threshold": threshold
    }

    classification_rows.append({
        "Function": function_number,
        "Threshold": threshold,
        "Class 0 count": int(class_counts[0]),
        "Class 1 count": int(class_counts[1]),
        "Round 3 class": int(labels[-1])
    })

classification_summary = pd.DataFrame(
    classification_rows
).set_index("Function")

display(
    classification_summary.style.format({
        "Threshold": "{:.6e}"
    })
)

,Threshold,Class 0 count,Class 1 count,Round 3 class
Function,,,,
1,6.793724e-80,7,6,1
2,2.297919e-01,6,7,1
3,-1.059650e-01,10,8,0
4,-1.604008e+01,15,18,1
5,6.394843e+01,10,13,1
6,-1.446370e+00,10,13,1
7,7.863144e-02,15,18,1
8,7.889150e+00,20,23,1


**Classification labels and class balance**

The thresholds remain fixed at the initial output medians. Each function has at least six observations in each class, allowing an exploratory three-fold stratified comparison.

Seven round-3 outputs belong to class 1, but only Functions 5 and 8 improved their previous maxima. Therefore, predicting class 1 is not equivalent to predicting an optimisation improvement. Function 1 remains particularly sensitive to the threshold because its labels distinguish extremely small values near zero.


In [8]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

# Fixed configurations for an initial model comparison
classifier_templates = {
    "Logistic regression": LogisticRegression(
        C=1.0,
        max_iter=2000,
        random_state=42
    ),
    "Linear SVM": SVC(
        kernel="linear",
        C=1.0
    ),
    "RBF SVM": SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale"
    ),
    "Neural network": MLPClassifier(
        hidden_layer_sizes=(8,),
        activation="tanh",
        solver="lbfgs",
        alpha=0.1,
        max_iter=2000,
        random_state=42
    )
}

classification_models = {}
comparison_rows = []

for function_number in range(1, 9):
    X = classification_data[function_number]["X"]
    labels = classification_data[function_number]["labels"]

    # Use exactly the same validation splits for all four models
    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )
    splits = list(cv.split(X, labels))

    classification_models[function_number] = {}

    for model_name, template in classifier_templates.items():
        # Scaling is fitted separately within each training fold
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", clone(template))
        ])

        scores = cross_validate(
            model,
            X,
            labels,
            cv=splits,
            scoring="balanced_accuracy",
            return_train_score=True,
            error_score="raise"
        )

        # Refit on all observations for later boundary analysis
        model.fit(X, labels)
        classification_models[function_number][model_name] = model

        comparison_rows.append({
            "Function": function_number,
            "Model": model_name,
            "Training balanced accuracy":
                scores["train_score"].mean(),
            "CV balanced accuracy":
                scores["test_score"].mean(),
            "CV standard deviation":
                scores["test_score"].std()
        })

classifier_comparison = pd.DataFrame(comparison_rows)

display(
    classifier_comparison.style.format({
        "Training balanced accuracy": "{:.3f}",
        "CV balanced accuracy": "{:.3f}",
        "CV standard deviation": "{:.3f}"
    })
)

,Function,Model,Training balanced accuracy,CV balanced accuracy,CV standard deviation
0,1,Logistic regression,0.933,0.944,0.079
1,1,Linear SVM,0.933,0.861,0.104
2,1,RBF SVM,0.933,0.778,0.171
3,1,Neural network,0.967,0.778,0.039
4,2,Logistic regression,0.725,0.639,0.157
5,2,Linear SVM,0.692,0.694,0.079
6,2,RBF SVM,0.833,0.833,0.118
7,2,Neural network,0.925,0.833,0.118
8,3,Logistic regression,0.769,0.667,0.136
9,3,Linear SVM,0.741,0.625,0.189


**Comparison of classification models**

The neural network achieved the highest mean cross-validation balanced accuracy for Functions 4 and 7 and tied with the RBF SVM for Function 2. Logistic regression performed best for Functions 1, 3 and 8, while the RBF SVM achieved the highest mean scores for Functions 5 and 6. These results do not show a consistent advantage from adding neural-network flexibility.

For Functions 3–8, the network achieved perfect average training accuracy but lower validation accuracy, indicating a potential generalisation problem. Several differences between models were small relative to the variation across folds, so the rankings remain tentative.

These scores describe classification relative to fixed initial-median thresholds. They do not measure the ability to locate a new maximum. In particular, Function 1's strong logistic-regression score does not resolve the practical limitations of its near-zero threshold.


In [9]:
# Identify support vectors near each fitted SVM decision boundary
selected_boundary_svms = {}
boundary_support_points = {}
support_rows = []

for function_number in range(1, 9):
    data = classification_data[function_number]
    X = data["X"]
    y = data["y"]
    labels = data["labels"]

    # Compare only the two SVM configurations
    svm_scores = classifier_comparison[
        (classifier_comparison["Function"] == function_number)
        & classifier_comparison["Model"].isin(
            ["Linear SVM", "RBF SVM"]
        )
    ]

    # Preserve the original order to favour linear SVM in an exact tie
    best_row = svm_scores.sort_values(
        "CV balanced accuracy",
        ascending=False,
        kind="stable"
    ).iloc[0]

    model_name = best_row["Model"]
    model = classification_models[function_number][model_name]
    selected_boundary_svms[function_number] = model

    # Indices refer to the original rows used to fit the pipeline
    support_indices = model.named_steps["classifier"].support_
    decision_scores = model.decision_function(X)

    # Select two actual support vectors closest to the fitted boundary
    order = np.argsort(np.abs(decision_scores[support_indices]))
    chosen_indices = support_indices[order[:2]]

    boundary_support_points[function_number] = {
        "indices": chosen_indices.copy(),
        "X": X[chosen_indices].copy()
    }

    initial_count = len(initial_data[function_number]["y"])

    for index in chosen_indices:
        source = (
            f"Initial observation {index + 1}"
            if index < initial_count
            else f"Round {index - initial_count + 1}"
        )

        support_rows.append({
            "Function": function_number,
            "SVM": model_name,
            "Observation": source,
            "Input": ", ".join(f"{v:.6f}" for v in X[index]),
            "Observed output": float(y[index]),
            "Actual class": int(labels[index]),
            "Predicted class": int(
                model.predict(X[index:index + 1])[0]
            ),
            "Decision score": float(decision_scores[index])
        })

support_vector_review = pd.DataFrame(support_rows)

with pd.option_context("display.max_colwidth", None):
    display(
        support_vector_review.style.format({
            "Observed output": "{:.6e}",
            "Decision score": "{:.4f}"
        })
    )

,Function,SVM,Observation,Input,Observed output,Actual class,Predicted class,Decision score
0,1,Linear SVM,Initial observation 4,"0.840353, 0.264732",3.341771e-124,0,0,-0.6889
1,1,Linear SVM,Round 3,"0.881159, 0.530910",1.210848e-51,1,1,0.7338
2,2,RBF SVM,Initial observation 7,"0.438166, 0.685018",2.446193e-01,1,1,0.2115
3,2,RBF SVM,Initial observation 5,"0.454647, 0.290455",2.149645e-01,0,0,-0.5465
4,3,Linear SVM,Initial observation 3,"0.534906, 0.398501, 0.173389",-1.114147e-01,0,0,-0.2529
5,3,Linear SVM,Initial observation 4,"0.492581, 0.611593, 0.340176",-3.483531e-02,1,1,0.3136
6,4,RBF SVM,Initial observation 13,"0.655115, 0.072392, 0.687152, 0.081517",-1.667912e+01,0,0,-0.0541
7,4,RBF SVM,Initial observation 10,"0.626071, 0.586751, 0.438806, 0.778858",-1.274177e+01,1,1,0.0768
8,5,RBF SVM,Initial observation 7,"0.553621, 0.667350, 0.323806, 0.814870",5.757154e+01,0,1,0.2208
9,5,RBF SVM,Initial observation 9,"0.153786, 0.729382, 0.422598, 0.443074",8.847992e+00,0,0,-0.2270


**Support vectors and boundary exploration**

The selected points are actual support vectors from the fitted SVMs, ranked within each model by their absolute decision scores. For Function 7, initial observation 23, at approximately (0.686853, 0.041017, 0.007573, 0.285009, 0.691568, 0.655543), has a decision score of 0.0063, placing it close to the fitted classification boundary.

Function 4's initial observations 13 and 10 lie on opposite sides of its fitted boundary, with scores of -0.0541 and 0.0768. Misclassified support vectors also appear in Functions 5, 6 and 8, illustrating that soft-margin models permit classification errors.

These observations identify locations around which additional sampling could clarify the classification boundary. However, proximity to this boundary does not imply proximity to a maximum or rapid change in the true objective. Decision-score magnitudes should not be compared directly across different SVMs.


In [10]:
# Compute input gradients for our binary, single-hidden-layer tanh network
def neural_class_probability_gradients(model, X):
    scaler = model.named_steps["scaler"]
    network = model.named_steps["classifier"]

    assert network.activation == "tanh"
    assert network.out_activation_ == "logistic"
    assert len(network.coefs_) == 2
    assert np.array_equal(network.classes_, [0, 1])

    X_scaled = scaler.transform(X)

    # Forward pass through the hidden layer
    W_hidden = network.coefs_[0]
    b_hidden = network.intercepts_[0]
    W_output = network.coefs_[1][:, 0]

    hidden = np.tanh(X_scaled @ W_hidden + b_hidden)
    probability = model.predict_proba(X)[:, 1]

    # Chain rule: output logit -> hidden activations -> scaled inputs
    logit_gradients_scaled = (
        (1.0 - hidden ** 2) * W_output
    ) @ W_hidden.T

    # Include the sigmoid derivative and undo input scaling
    probability_gradients = (
        probability[:, None]
        * (1.0 - probability[:, None])
        * logit_gradients_scaled
        / scaler.scale_[None, :]
    )

    return probability, probability_gradients


classification_gradients = {}
gradient_rows = []

for function_number in range(1, 9):
    X = classification_data[function_number]["X"]
    model = classification_models[function_number]["Neural network"]

    probability, gradients = neural_class_probability_gradients(
        model, X
    )

    # Summarise sensitivity at the observed inputs
    mean_absolute_gradient = np.mean(np.abs(gradients), axis=0)

    # Examine sensitivity at the two selected SVM support vectors
    boundary_indices = boundary_support_points[
        function_number
    ]["indices"]

    boundary_mean_absolute_gradient = np.mean(
        np.abs(gradients[boundary_indices]),
        axis=0
    )

    classification_gradients[function_number] = {
        "probabilities": probability,
        "gradients": gradients,
        "mean_absolute_gradient": mean_absolute_gradient,
        "boundary_mean_absolute_gradient":
            boundary_mean_absolute_gradient
    }

    for variable_index in range(X.shape[1]):
        gradient_rows.append({
            "Function": function_number,
            "Variable": f"x{variable_index + 1}",
            "Mean absolute probability gradient":
                mean_absolute_gradient[variable_index],
            "At selected support vectors":
                boundary_mean_absolute_gradient[variable_index]
        })

classification_gradient_summary = pd.DataFrame(gradient_rows)

display(
    classification_gradient_summary.style.format({
        "Mean absolute probability gradient": "{:.4f}",
        "At selected support vectors": "{:.4f}"
    })
)

,Function,Variable,Mean absolute probability gradient,At selected support vectors
0,1,x1,0.3280,0.6417
1,1,x2,0.7824,1.5306
2,2,x1,0.8598,1.2577
3,2,x2,0.6379,0.1152
4,3,x1,0.3352,0.5415
5,3,x2,0.3655,0.6376
6,3,x3,1.1747,1.8524
7,4,x1,0.1301,0.3022
8,4,x2,0.4426,1.1342
9,4,x3,0.2690,0.7481


**Neural-network sensitivity to input variables**

The largest mean absolute probability gradients occurred for x2 in Function 1, x1 in Function 2, x3 in Function 3 and x2 in Function 4. Function 5 showed similar average sensitivity to x1, x2 and x4, while Functions 6 and 7 had relatively small differences between their leading variables.

Sensitivity also depended on location. In Function 8, x3 had the largest average gradient across observed inputs, whereas x7 was most influential at the selected SVM support vectors. In Function 7, x5 was most influential at those support vectors despite x3 having the largest overall average.

These findings suggest which coordinates could be varied in local boundary experiments. However, they describe the fitted classifier's probability of exceeding the initial median, not changes in the numerical objective. Absolute gradients also discard direction, so they cannot tell us whether increasing a coordinate would improve performance.


In [11]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import KFold

# Fixed configurations for numerical output prediction
regressor_templates = {
    "Training mean": DummyRegressor(strategy="mean"),
    "Linear regression": LinearRegression(),
    "Neural network": MLPRegressor(
        hidden_layer_sizes=(8,),
        activation="tanh",
        solver="lbfgs",
        alpha=0.1,
        max_iter=2000,
        random_state=42
    )
}

regression_models = {}
regression_rows = []

for function_number in range(1, 9):
    X = updated_data[function_number]["X"]
    y = updated_data[function_number]["y"]

    # Use the same splits for all regressors within each function
    cv = KFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )
    splits = list(cv.split(X))

    regression_models[function_number] = {}

    for model_name, template in regressor_templates.items():
        input_pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("regressor", clone(template))
        ])

        # Target scaling is also fitted separately within each fold
        model = TransformedTargetRegressor(
            regressor=input_pipeline,
            transformer=StandardScaler()
        )

        scores = cross_validate(
            model,
            X,
            y,
            cv=splits,
            scoring="neg_mean_squared_error",
            return_train_score=True,
            error_score="raise"
        )

        train_rmse = np.sqrt(-scores["train_score"])
        validation_rmse = np.sqrt(-scores["test_score"])

        # Refit on all observations for later gradient analysis
        model.fit(X, y)
        regression_models[function_number][model_name] = model

        regression_rows.append({
            "Function": function_number,
            "Model": model_name,
            "Mean training RMSE": train_rmse.mean(),
            "Mean CV RMSE": validation_rmse.mean(),
            "CV RMSE standard deviation": validation_rmse.std()
        })

regression_comparison = pd.DataFrame(regression_rows)

display(
    regression_comparison.style.format({
        "Mean training RMSE": "{:.6e}",
        "Mean CV RMSE": "{:.6e}",
        "CV RMSE standard deviation": "{:.6e}"
    })
)

,Function,Model,Mean training RMSE,Mean CV RMSE,CV RMSE standard deviation
0,1,Training mean,7.755361e-04,8.846920e-04,6.495697e-04
1,1,Linear regression,7.577604e-04,9.258115e-04,6.257285e-04
2,1,Neural network,1.774796e-04,2.329723e-03,5.389715e-04
3,2,Training mean,2.390199e-01,2.379322e-01,3.723101e-02
4,2,Linear regression,1.894809e-01,2.133448e-01,8.917054e-02
5,2,Neural network,5.108463e-02,2.358895e-01,2.867071e-02
6,3,Training mean,7.743103e-02,7.582720e-02,3.624259e-02
7,3,Linear regression,5.795323e-02,8.663583e-02,2.985677e-02
8,3,Neural network,1.051035e-02,8.025822e-02,3.282141e-02
9,4,Training mean,8.036968e+00,8.290367e+00,2.471900e+00


**Regression performance and model complexity**

The neural network achieved the lowest mean cross-validation RMSE among the three tested regressors for Functions 4, 5, 6 and 8. Its advantage over linear regression was substantial for Functions 4 and 5, while the difference for Function 6 was relatively small. Linear regression performed best for Function 2, and the training-mean baseline performed best for Functions 1, 3 and 7.

Functions 1 and 7 showed particularly large gaps between the network's training and validation errors. This suggests overfitting and argues against using their neural-network predictions as the main query-selection criterion. Function 7 also illustrates that reasonable classification performance does not guarantee accurate numerical output predictions.

These comparisons are exploratory and depend on a small data set, three validation folds and one network initialisation. They do not yet establish whether the neural network outperforms the GP, which was not included in this regression comparison.


In [12]:
# Compute input gradients for the fitted neural-network regressor
def neural_output_gradients(model, X):
    # Access fitted components of TransformedTargetRegressor
    input_pipeline = model.regressor_
    input_scaler = input_pipeline.named_steps["scaler"]
    network = input_pipeline.named_steps["regressor"]
    target_scaler = model.transformer_

    assert network.activation == "tanh"
    assert network.out_activation_ == "identity"
    assert len(network.coefs_) == 2

    X_scaled = input_scaler.transform(X)

    W_hidden = network.coefs_[0]
    b_hidden = network.intercepts_[0]
    W_output = network.coefs_[1][:, 0]

    # Forward pass through the hidden layer
    hidden = np.tanh(X_scaled @ W_hidden + b_hidden)

    # Chain rule for the standardised numerical output
    gradients_scaled = (
        (1.0 - hidden ** 2) * W_output
    ) @ W_hidden.T

    # Express gradients in original input and output units
    gradients = (
        gradients_scaled
        * target_scaler.scale_[0]
        / input_scaler.scale_[None, :]
    )

    return model.predict(X), gradients


regression_gradients = {}
output_gradient_rows = []

for function_number in range(1, 9):
    X = updated_data[function_number]["X"]
    y = updated_data[function_number]["y"]
    model = regression_models[function_number]["Neural network"]

    predictions, gradients = neural_output_gradients(model, X)

    best_index = int(np.argmax(y))
    mean_absolute_gradient = np.mean(np.abs(gradients), axis=0)
    gradient_at_best = gradients[best_index]

    regression_gradients[function_number] = {
        "predictions": predictions,
        "gradients": gradients,
        "mean_absolute_gradient": mean_absolute_gradient,
        "best_index": best_index,
        "best_input": X[best_index].copy(),
        "gradient_at_best": gradient_at_best.copy()
    }

    for variable_index in range(X.shape[1]):
        output_gradient_rows.append({
            "Function": function_number,
            "Variable": f"x{variable_index + 1}",
            "Mean absolute output gradient":
                mean_absolute_gradient[variable_index],
            "Signed gradient at best observed input":
                gradient_at_best[variable_index]
        })

output_gradient_summary = pd.DataFrame(output_gradient_rows)

display(
    output_gradient_summary.style.format({
        "Mean absolute output gradient": "{:.6e}",
        "Signed gradient at best observed input": "{:.6e}"
    })
)

,Function,Variable,Mean absolute output gradient,Signed gradient at best observed input
0,1,x1,5.913134e-03,2.882765e-02
1,1,x2,5.848064e-03,1.928620e-02
2,2,x1,8.504343e-01,-1.735980e-01
3,2,x2,5.158974e-01,-4.825617e-01
4,3,x1,1.691822e-01,-9.861961e-02
5,3,x2,1.409871e-01,-8.917444e-02
6,3,x3,2.503597e-01,8.058015e-02
7,4,x1,1.254058e+01,4.074393e-01
8,4,x2,1.928225e+01,7.284709e-02
9,4,x3,1.261846e+01,-1.232585e+00


**Numerical-output gradients and candidate directions**

For Function 5, x4 and x3 had the largest mean absolute output gradients. All four signed gradients were positive at the best observed input, suggesting local increases according to the network. However, several coordinates are already close to their upper bounds, which limits feasible movement.

For Function 4, x2 had the largest average sensitivity, but decreasing x3 was the strongest local direction at the best observed point. Function 6 showed the greatest average sensitivity to x5 and x3, while its strongest local direction was to decrease x3. For Function 8, x3 dominated average sensitivity, whereas x1 had the largest absolute gradient at the best observed input.

These differences demonstrate that average sensitivity and local search directions answer different questions. I will use small, bounded gradient steps to generate supplementary candidates for Functions 4, 5, 6 and 8. I will not use the neural-network gradients as the main guide for Functions 1 and 7 because their regression validation results were poor. None of these gradients establishes how the unknown function will actually respond.


In [13]:
# Generate supplementary candidates using local neural-network gradients
gradient_functions = [4, 5, 6, 8]
step_sizes = [0.01, 0.03, 0.05]

gradient_candidates = {}
gradient_candidate_rows = []

for function_number in gradient_functions:
    X = updated_data[function_number]["X"]
    y = updated_data[function_number]["y"]
    model = regression_models[function_number]["Neural network"]

    best_index = int(np.argmax(y))
    best_input = X[best_index].copy()

    gradient = regression_gradients[
        function_number
    ]["gradient_at_best"]

    gradient_norm = np.linalg.norm(gradient)

    if not np.isfinite(gradient_norm) or gradient_norm == 0:
        gradient_candidates[function_number] = np.empty(
            (0, X.shape[1])
        )
        continue

    # Normalise direction so step sizes do not depend on output units
    direction = gradient / gradient_norm

    # Compare predictions at candidates with the prediction at the start
    start_prediction = float(
        model.predict(best_input.reshape(1, -1))[0]
    )

    accepted_candidates = []

    for step_size in step_sizes:
        # Take one ascent step from the best observed input
        candidate = best_input + step_size * direction

        # Respect portal bounds and precision
        candidate = np.round(
            np.clip(candidate, 0.0, 0.999999),
            6
        )

        nearest_distance = float(
            np.min(np.linalg.norm(X - candidate, axis=1))
        )

        if nearest_distance <= 1e-6:
            continue

        if any(
            np.array_equal(candidate, previous)
            for previous in accepted_candidates
        ):
            continue

        prediction = float(
            model.predict(candidate.reshape(1, -1))[0]
        )

        accepted_candidates.append(candidate.copy())

        gradient_candidate_rows.append({
            "Function": function_number,
            "Requested step": step_size,
            "Actual movement": float(
                np.linalg.norm(candidate - best_input)
            ),
            "Candidate": "-".join(
                f"{value:.6f}" for value in candidate
            ),
            "NN prediction at start": start_prediction,
            "NN prediction at candidate": prediction,
            "NN predicted change": prediction - start_prediction,
            "Nearest observation distance": nearest_distance
        })

    gradient_candidates[function_number] = np.asarray(
        accepted_candidates,
        dtype=float
    ).reshape(-1, X.shape[1])

gradient_candidate_summary = pd.DataFrame(
    gradient_candidate_rows
)

display(
    gradient_candidate_summary.style.format({
        "Requested step": "{:.2f}",
        "Actual movement": "{:.6f}",
        "NN prediction at start": "{:.6e}",
        "NN prediction at candidate": "{:.6e}",
        "NN predicted change": "{:.6e}",
        "Nearest observation distance": "{:.6f}"
    })
)

,Function,Requested step,Actual movement,Candidate,NN prediction at start,NN prediction at candidate,NN predicted change,Nearest observation distance
0,4,0.01,0.010000,0.406953-0.420641-0.347836-0.444432,-1.030997e+00,-1.019395e+00,1.160165e-02,0.010000
1,4,0.03,0.030000,0.412452-0.421624-0.331202-0.434835,-1.030997e+00,-1.015234e+00,1.576287e-02,0.030000
2,4,0.05,0.050000,0.417950-0.422607-0.314567-0.425238,-1.030997e+00,-1.035858e+00,-4.860804e-03,0.039104
3,5,0.01,0.008314,0.509392-0.999999-0.976763-0.999999,4.105585e+03,4.165670e+03,6.008441e+01,0.008314
4,5,0.03,0.021422,0.517033-0.999999-0.988446-0.999999,4.105585e+03,4.237392e+03,1.318067e+02,0.021422
5,5,0.05,0.035083,0.524674-0.999999-0.999999-0.999999,4.105585e+03,4.305214e+03,1.996288e+02,0.035083
6,6,0.01,0.010000,0.448668-0.360439-0.324487-0.899561-0.004934,-6.260200e-01,-6.134007e-01,1.261934e-02,0.010000
7,6,0.03,0.030000,0.446724-0.348862-0.309323-0.893896-0.005331,-6.260200e-01,-5.890743e-01,3.694571e-02,0.030000
8,6,0.05,0.050000,0.444781-0.337286-0.294158-0.888231-0.005727,-6.260200e-01,-5.661048e-01,5.991524e-02,0.050000
9,8,0.01,0.010000,0.007684-0.037118-0.061003-0.040351-0.941396-0.627859-0.023128-0.943083,9.894178e+00,9.900463e+00,6.284476e-03,0.010000


**Gradient-based candidate generation**

Small gradient-ascent steps generated supplementary candidates for Functions 4, 5, 6 and 8. For Function 4, steps of 0.01 and 0.03 increased the network's prediction, but the 0.05 step decreased it. This illustrates that a local ascent direction does not guarantee improvement for every finite step. The network also underestimated the output at the starting point, predicting approximately -1.031 compared with the observed value of 0.195.

The network predicted increases for all three candidates in Functions 5, 6 and 8. Clipping reduced the actual movement for some candidates in Functions 5 and 8 because coordinates reached the permitted boundaries.

These changes compare predictions within the same neural network, not actual function evaluations. The candidates will be assessed alongside globally sampled points using the GP acquisition function. They remain supplementary proposals rather than confirmed improvements or final submissions.


In [14]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# Use all Function 1 observations available for round 4
X_1 = updated_data[1]["X"]
y_1 = updated_data[1]["y"]

models_1 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_1.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_1, y_1)
    models_1[name] = model

    print(f"\nModel: {name}")
    print("Observations:", len(y_1))
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )

c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 0.02. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



Model: A_original
Observations: 13
Fitted kernel: 1.07**2 * Matern(length_scale=[0.02, 2], nu=2.5) + WhiteKernel(noise_level=6.67e-10)
Log-marginal likelihood: -16.83450702410689

Model: B_wider
Observations: 13
Fitted kernel: 1.07**2 * Matern(length_scale=[0.02, 10], nu=2.5) + WhiteKernel(noise_level=3.43e-09)
Log-marginal likelihood: -16.55548818270537


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 0.02. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


**Function 1: updated GP fit and sensitivity to bounds**

Both GP models fitted the 13 available observations. The wider-bound model improved log-marginal likelihood by approximately 0.279.

The fitted length-scale pattern changed substantially from the previous round: x1 now reaches the lower bound of 0.02, while x2 reaches the upper bound. This reversal suggests that the inferred directional behaviour is sensitive to the available data and hyperparameter optimisation. It should not be interpreted as definitive feature importance.

Before selecting a query, I will check whether allowing a shorter length scale materially changes the fit. This is a targeted sensitivity check prompted by the new lower-bound warning.


In [15]:
# Test a smaller lower bound, starting from the fitted B model
kernel_1_sensitivity = clone(models_1["B_wider"].kernel_)

kernel_1_sensitivity.set_params(
    k1__k2__length_scale_bounds=(0.002, 10.0)
)

model_1_sensitivity = GaussianProcessRegressor(
    kernel=kernel_1_sensitivity,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

model_1_sensitivity.fit(X_1, y_1)
models_1["C_lower_bound"] = model_1_sensitivity

# Compare all three fits on the same observations
for name, model in models_1.items():
    print(f"\nModel: {name}")
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )


Model: A_original
Fitted kernel: 1.07**2 * Matern(length_scale=[0.02, 2], nu=2.5) + WhiteKernel(noise_level=6.67e-10)
Log-marginal likelihood: -16.83450702410689

Model: B_wider
Fitted kernel: 1.07**2 * Matern(length_scale=[0.02, 10], nu=2.5) + WhiteKernel(noise_level=3.43e-09)
Log-marginal likelihood: -16.55548818270537

Model: C_lower_bound
Fitted kernel: 1.03**2 * Matern(length_scale=[0.0168, 10], nu=2.5) + WhiteKernel(noise_level=3.43e-09)
Log-marginal likelihood: -16.44613536788407


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


**Function 1: lower-bound sensitivity check**

Allowing shorter length scales produced a fitted value of approximately 0.0168 for x1, which lies inside the revised bounds. The previous lower bound of 0.02 therefore constrained the fit slightly. Log-marginal likelihood improved by approximately 0.109 relative to B_wider.

The second length scale still reached its upper bound of 10. This remaining warning does not, by itself, justify further expansion or establish that x2 is irrelevant. I will compare the three models' proposed queries to assess how much these modelling choices affect the search.


In [16]:
# Generate the same two-dimensional candidates for all three GP models
rng_1 = np.random.default_rng(42)
candidates_1 = rng_1.uniform(
    0, 0.999999, size=(100000, X_1.shape[1])
)

# Round to portal-ready coordinates
candidates_1 = np.unique(np.round(candidates_1, 6), axis=0)

# Calculate distance to the nearest existing observation
nearest_distance_1 = np.min(
    np.linalg.norm(
        candidates_1[:, None, :] - X_1[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude previously evaluated points
keep = nearest_distance_1 > 1e-6
candidates_1 = candidates_1[keep]
nearest_distance_1 = nearest_distance_1[keep]

# Compare UCB proposals without filtering by classification labels
beta_1 = 2.0
proposals_1 = {}

for name, model in models_1.items():
    mean, std = model.predict(candidates_1, return_std=True)
    ucb = mean + beta_1 * std

    index = np.argmax(ucb)
    query = candidates_1[index].copy()

    svm_class = int(
        selected_boundary_svms[1].predict(
            query.reshape(1, -1)
        )[0]
    )

    proposals_1[name] = {
        "query": query,
        "mean": float(mean[index]),
        "std": float(std[index]),
        "ucb": float(ucb[index]),
        "nearest_distance": float(nearest_distance_1[index]),
        "svm_class": svm_class,
        "candidate_source": "Random search"
    }

    print(f"\nModel: {name}")
    print("Proposed query:", "-".join(f"{x:.6f}" for x in query))
    print(f"Predicted output: {mean[index]:.6e}")
    print(f"Predictive standard deviation: {std[index]:.6e}")
    print(f"UCB score: {ucb[index]:.6e}")
    print(
        f"Distance to nearest observation: "
        f"{nearest_distance_1[index]:.6f}"
    )
    print("SVM class (1 = above threshold):", svm_class)


Model: A_original
Proposed query: 0.704931-0.003150
Predicted output: 1.204577e-04
Predictive standard deviation: 8.663607e-04
UCB score: 1.853179e-03
Distance to nearest observation: 0.222743
SVM class (1 = above threshold): 0

Model: B_wider
Proposed query: 0.704931-0.003150
Predicted output: 1.677763e-04
Predictive standard deviation: 8.354883e-04
UCB score: 1.838753e-03
Distance to nearest observation: 0.222743
SVM class (1 = above threshold): 0

Model: C_lower_bound
Proposed query: 0.705334-0.019831
Predicted output: 1.038813e-05
Predictive standard deviation: 8.857243e-04
UCB score: 1.781837e-03
Distance to nearest observation: 0.207724
SVM class (1 = above threshold): 0


**Function 1: query comparison and selection**

All three GP models proposed a similar region, with x1 around 0.705 and a low value of x2. I selected C_lower_bound because it relaxed a binding lower constraint and achieved the highest log-marginal likelihood, although its advantage over B_wider was small.

The selected query is primarily exploratory: its predicted standard deviation is much larger than its predicted mean, so the uncertainty term dominates UCB. The SVM assigned class 0 to all three candidates. I recorded this disagreement without excluding the region, since the near-zero classification threshold has limited practical meaning for optimisation. The proposal is not evidence of a positive true output or a new maximum.


In [17]:
# Store the selected model and query for Function 1, round 4
selected_model_1 = models_1["C_lower_bound"]
query_1_round4 = proposals_1["C_lower_bound"]["query"].copy()

# Preserve prediction details for comparison with future portal feedback
selected_proposal_1_round4 = proposals_1["C_lower_bound"].copy()

print(
    "Function 1 — round 4:",
    "-".join(f"{x:.6f}" for x in query_1_round4)
)

Function 1 — round 4: 0.705334-0.019831


**Stored selection**

The selected GP, query coordinates and prediction details have been stored for Function 1, round 4. The portal-ready query is 0.705334-0.019831. Its actual output will only be known after evaluation by the portal.


In [18]:
# Use all Function 2 observations available for round 4
X_2 = updated_data[2]["X"]
y_2 = updated_data[2]["y"]

models_2 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_2.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_2, y_2)
    models_2[name] = model

    print(f"\nModel: {name}")
    print("Observations:", len(y_2))
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )

c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



Model: A_original
Observations: 13
Fitted kernel: 0.926**2 * Matern(length_scale=[0.056, 2], nu=2.5) + WhiteKernel(noise_level=0.01)
Log-marginal likelihood: -13.446047313791944

Model: B_wider
Observations: 13
Fitted kernel: 0.97**2 * Matern(length_scale=[0.0604, 3.97], nu=2.5) + WhiteKernel(noise_level=0.014)
Log-marginal likelihood: -13.208677436052536


**Function 2: updated GP comparison**

Both models fitted the 13 available observations. A_original reached the upper length-scale bound of 2 for x2, whereas B_wider fitted approximately 3.97, inside its permitted interval. The wider bounds therefore allowed a solution without this particular constraint being active.

B_wider improved log-marginal likelihood by approximately 0.237. This is a modest improvement in the fitting criterion, not proof of better predictive performance. I will compare the resulting query proposals before selecting the next point.


In [19]:
# Generate the same two-dimensional candidates for both GP models
rng_2 = np.random.default_rng(42)
candidates_2 = rng_2.uniform(
    0, 0.999999, size=(100000, X_2.shape[1])
)

# Round to portal-ready coordinates
candidates_2 = np.unique(np.round(candidates_2, 6), axis=0)

# Calculate distance to the nearest existing observation
nearest_distance_2 = np.min(
    np.linalg.norm(
        candidates_2[:, None, :] - X_2[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude previously evaluated points
keep = nearest_distance_2 > 1e-6
candidates_2 = candidates_2[keep]
nearest_distance_2 = nearest_distance_2[keep]

# Compare UCB proposals without filtering by classification labels
beta_2 = 2.0
proposals_2 = {}

for name, model in models_2.items():
    mean, std = model.predict(candidates_2, return_std=True)
    ucb = mean + beta_2 * std

    index = np.argmax(ucb)
    query = candidates_2[index].copy()

    svm_class = int(
        selected_boundary_svms[2].predict(
            query.reshape(1, -1)
        )[0]
    )

    proposals_2[name] = {
        "query": query,
        "mean": float(mean[index]),
        "std": float(std[index]),
        "ucb": float(ucb[index]),
        "nearest_distance": float(nearest_distance_2[index]),
        "svm_class": svm_class,
        "candidate_source": "Random search"
    }

    print(f"\nModel: {name}")
    print("Proposed query:", "-".join(f"{x:.6f}" for x in query))
    print(f"Predicted output: {mean[index]:.6e}")
    print(f"Predictive standard deviation: {std[index]:.6e}")
    print(f"UCB score: {ucb[index]:.6e}")
    print(
        f"Distance to nearest observation: "
        f"{nearest_distance_2[index]:.6f}"
    )
    print("SVM class (1 = above threshold):", svm_class)


Model: A_original
Proposed query: 0.697070-0.000388
Predicted output: 6.671011e-01
Predictive standard deviation: 6.594317e-02
UCB score: 7.989874e-01
Distance to nearest observation: 0.023854
SVM class (1 = above threshold): 1

Model: B_wider
Proposed query: 0.699810-0.002104
Predicted output: 6.634182e-01
Predictive standard deviation: 5.500270e-02
UCB score: 7.734236e-01
Distance to nearest observation: 0.020943
SVM class (1 = above threshold): 1


**Function 2: query comparison and selection**

Both GP models proposed points close to the best observed input. I selected B_wider because its fitted length scales lie inside the permitted bounds and its log-marginal likelihood was slightly higher.

The selected candidate has a predicted mean of approximately 0.6634, compared with the observed maximum of 0.6582. This small predicted improvement is modest relative to the predictive standard deviation of 0.0550. The query therefore emphasises local exploitation while retaining the UCB uncertainty bonus. Both candidates received SVM class 1, which does not establish that either will improve the current maximum.


In [20]:
# Store the selected model and query for Function 2, round 4
selected_model_2 = models_2["B_wider"]
query_2_round4 = proposals_2["B_wider"]["query"].copy()

# Preserve prediction details for comparison with future portal feedback
selected_proposal_2_round4 = proposals_2["B_wider"].copy()

print(
    "Function 2 — round 4:",
    "-".join(f"{x:.6f}" for x in query_2_round4)
)

Function 2 — round 4: 0.699810-0.002104


**Stored selection**

The selected GP, query coordinates and prediction details have been stored for Function 2, round 4. The portal-ready query is 0.699810-0.002104. Its actual performance remains unknown until the portal returns the evaluation.


In [21]:
# Use all Function 3 observations available for round 4
X_3 = updated_data[3]["X"]
y_3 = updated_data[3]["y"]

models_3 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_3.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_3, y_3)
    models_3[name] = model

    print(f"\nModel: {name}")
    print("Observations:", len(y_3))
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )

c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



Model: A_original
Observations: 18
Fitted kernel: 1.14**2 * Matern(length_scale=[2, 1.7, 0.0579], nu=2.5) + WhiteKernel(noise_level=1e-06)
Log-marginal likelihood: -20.761016456467438

Model: B_wider
Observations: 18
Fitted kernel: 1.24**2 * Matern(length_scale=[10, 1.43, 0.0724], nu=2.5) + WhiteKernel(noise_level=3.12e-06)
Log-marginal likelihood: -20.220954225956174


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


**Function 3: updated GP comparison**

B_wider improved log-marginal likelihood by approximately 0.540 relative to A_original. Both models reached the upper length-scale bound for x1, while the fitted length scale for x2 now lies inside the permitted interval. The third coordinate retains a much shorter fitted length scale.

These estimates suggest different rates of variation across coordinates within the fitted models, but they do not establish definitive feature importance. I will compare the proposed queries without further expanding the bounds. Neural-network gradient candidates are excluded for this function because the network did not outperform the training-mean baseline in regression validation.


In [23]:
# Generate the same three-dimensional candidates for both GP models
rng_3 = np.random.default_rng(42)
candidates_3 = rng_3.uniform(
    0, 0.999999, size=(100000, X_3.shape[1])
)

# Round to portal-ready coordinates
candidates_3 = np.unique(np.round(candidates_3, 6), axis=0)

# Calculate distance to the nearest existing observation
nearest_distance_3 = np.min(
    np.linalg.norm(
        candidates_3[:, None, :] - X_3[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude previously evaluated points
keep = nearest_distance_3 > 1e-6
candidates_3 = candidates_3[keep]
nearest_distance_3 = nearest_distance_3[keep]

# Compare UCB proposals without filtering by classification labels
beta_3 = 2.0
proposals_3 = {}

for name, model in models_3.items():
    mean, std = model.predict(candidates_3, return_std=True)
    ucb = mean + beta_3 * std

    index = np.argmax(ucb)
    query = candidates_3[index].copy()

    svm_class = int(
        selected_boundary_svms[3].predict(
            query.reshape(1, -1)
        )[0]
    )

    proposals_3[name] = {
        "query": query,
        "mean": float(mean[index]),
        "std": float(std[index]),
        "ucb": float(ucb[index]),
        "nearest_distance": float(nearest_distance_3[index]),
        "svm_class": svm_class,
        "candidate_source": "Random search"
    }

    print(f"\nModel: {name}")
    print("Proposed query:", "-".join(f"{x:.6f}" for x in query))
    print(f"Predicted output: {mean[index]:.6e}")
    print(f"Predictive standard deviation: {std[index]:.6e}")
    print(f"UCB score: {ucb[index]:.6e}")
    print(
        f"Distance to nearest observation: "
        f"{nearest_distance_3[index]:.6f}"
    )
    print("SVM class (1 = above threshold):", svm_class)


Model: A_original
Proposed query: 0.947068-0.977811-0.502363
Predicted output: -5.686131e-02
Predictive standard deviation: 6.914880e-02
UCB score: 8.143630e-02
Distance to nearest observation: 0.118670
SVM class (1 = above threshold): 1

Model: B_wider
Proposed query: 0.933505-0.993849-0.852342
Predicted output: -6.706418e-02
Predictive standard deviation: 7.876711e-02
UCB score: 9.047005e-02
Distance to nearest observation: 0.316528
SVM class (1 = above threshold): 1


**Function 3: query comparison and selection**

I retained B_wider because it achieved a modestly higher log-marginal likelihood and maintains continuity with the previous round's configuration. However, the models proposed substantially different third coordinates: approximately 0.502 for A and 0.852 for B. The query is therefore sensitive to the modelling bounds.

The selected candidate's predicted mean of approximately -0.0671 is below the current best output of -0.0348. Its uncertainty bonus makes it attractive under UCB, giving the query a substantial exploratory component. Both candidates received SVM class 1, which does not distinguish between them or guarantee an improvement.


In [24]:
# Store the selected model and query for Function 3, round 4
selected_model_3 = models_3["B_wider"]
query_3_round4 = proposals_3["B_wider"]["query"].copy()

# Preserve prediction details for comparison with future portal feedback
selected_proposal_3_round4 = proposals_3["B_wider"].copy()

print(
    "Function 3 — round 4:",
    "-".join(f"{x:.6f}" for x in query_3_round4)
)

Function 3 — round 4: 0.933505-0.993849-0.852342


**Stored selection**

The selected GP, query coordinates and prediction details have been stored for Function 3, round 4. The portal-ready query is 0.933505-0.993849-0.852342. Its actual output remains unknown until evaluation.


In [25]:
# Use all Function 4 observations available for round 4
X_4 = updated_data[4]["X"]
y_4 = updated_data[4]["y"]

models_4 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_4.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_4, y_4)
    models_4[name] = model

    print(f"\nModel: {name}")
    print("Observations:", len(y_4))
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )


Model: A_original
Observations: 33
Fitted kernel: 2.89**2 * Matern(length_scale=[1.61, 1.45, 1.52, 1.47], nu=2.5) + WhiteKernel(noise_level=0.000935)
Log-marginal likelihood: -13.223434731900642

Model: B_wider
Observations: 33
Fitted kernel: 2.89**2 * Matern(length_scale=[1.61, 1.45, 1.52, 1.47], nu=2.5) + WhiteKernel(noise_level=0.000935)
Log-marginal likelihood: -13.223434731900138


**Function 4 — Gaussian Process comparison**

Both kernel configurations converged to virtually identical fitted parameters and log-marginal likelihoods. All four length scales remained below the original upper bound of 2.0, so widening this bound did not materially change the fitted model. I retained A_original.

Next, I will compare random candidates with the local candidates generated using neural network gradients. The Gaussian Process UCB score will rank both groups using the same criterion. This allows the neural network to contribute candidate locations without assuming that its predicted improvements will occur in the actual function.

In [26]:
# Retain the original GP configuration for Function 4
selected_model_4 = models_4["A_original"]

# Generate random candidates
rng_4 = np.random.default_rng(42)
random_candidates_4 = rng_4.uniform(
    0, 0.999999, size=(100000, X_4.shape[1])
)
random_candidates_4 = np.unique(
    np.round(random_candidates_4, 6), axis=0
)

# Include the candidates previously generated using NN gradients
nn_candidates_4 = np.asarray(gradient_candidates[4])

# Combine both candidate groups and remove duplicates
candidates_4 = np.unique(
    np.vstack([random_candidates_4, nn_candidates_4]),
    axis=0
)

# Identify candidates originating from the NN gradient experiment
is_nn_candidate_4 = np.any(
    np.all(
        candidates_4[:, None, :] == nn_candidates_4[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude previously evaluated points
nearest_distance_4 = np.min(
    np.linalg.norm(
        candidates_4[:, None, :] - X_4[None, :, :],
        axis=2
    ),
    axis=1
)

keep_4 = nearest_distance_4 > 1e-6
candidates_4 = candidates_4[keep_4]
is_nn_candidate_4 = is_nn_candidate_4[keep_4]
nearest_distance_4 = nearest_distance_4[keep_4]

# Evaluate every candidate with the same GP acquisition function
beta_4 = 2.0
mean_4, std_4 = selected_model_4.predict(
    candidates_4, return_std=True
)
ucb_4 = mean_4 + beta_4 * std_4

# Display the best random candidate and all NN gradient candidates
random_indices_4 = np.flatnonzero(~is_nn_candidate_4)
best_random_index_4 = random_indices_4[
    np.argmax(ucb_4[random_indices_4])
]
nn_indices_4 = np.flatnonzero(is_nn_candidate_4)

comparison_rows_4 = []

for index in [best_random_index_4, *nn_indices_4]:
    comparison_rows_4.append({
        "Source": (
            "NN gradient"
            if is_nn_candidate_4[index]
            else "Best random candidate"
        ),
        "Query": "-".join(
            f"{x:.6f}" for x in candidates_4[index]
        ),
        "GP predicted mean": mean_4[index],
        "GP predicted std": std_4[index],
        "UCB": ucb_4[index],
        "Nearest observation distance": nearest_distance_4[index]
    })

display(
    pd.DataFrame(comparison_rows_4)
    .sort_values("UCB", ascending=False)
    .reset_index(drop=True)
)

# Store the selected model's highest-UCB query for round 4
best_index_4 = int(np.argmax(ucb_4))
query_4_round4 = candidates_4[best_index_4].copy()

svm_class_4 = int(
    selected_boundary_svms[4].predict(
        query_4_round4.reshape(1, -1)
    )[0]
)

selected_proposal_4_round4 = {
    "query": query_4_round4.copy(),
    "mean": float(mean_4[best_index_4]),
    "std": float(std_4[best_index_4]),
    "ucb": float(ucb_4[best_index_4]),
    "nearest_distance": float(nearest_distance_4[best_index_4]),
    "svm_class": svm_class_4,
    "candidate_source": (
        "NN gradient"
        if is_nn_candidate_4[best_index_4]
        else "Random search"
    )
}

print(
    "\nFunction 4 — round 4:",
    "-".join(f"{x:.6f}" for x in query_4_round4)
)
print("Selected model: A_original")
print("Candidate source:", selected_proposal_4_round4["candidate_source"])
print(f"Predicted output: {mean_4[best_index_4]:.6e}")
print(f"Predictive standard deviation: {std_4[best_index_4]:.6e}")
print(f"UCB score: {ucb_4[best_index_4]:.6e}")
print(f"Distance to nearest observation: {nearest_distance_4[best_index_4]:.6f}")
print("SVM class (1 = above threshold):", svm_class_4)

,Source,Query,GP predicted mean,GP predicted std,UCB,Nearest observation distance
0,Best random candidate,0.367088-0.474599-0.423453-0.415817,-0.390458,0.467940,0.545421,0.099941
1,NN gradient,0.406953-0.420641-0.347836-0.444432,-0.150517,0.304183,0.457849,0.010000
2,NN gradient,0.412452-0.421624-0.331202-0.434835,-0.256942,0.306135,0.355328,0.030000
3,NN gradient,0.417950-0.422607-0.314567-0.425238,-0.416762,0.314165,0.211568,0.039104



Function 4 — round 4: 0.367088-0.474599-0.423453-0.415817
Selected model: A_original
Candidate source: Random search
Predicted output: -3.904580e-01
Predictive standard deviation: 4.679397e-01
UCB score: 5.454214e-01
Distance to nearest observation: 0.099941
SVM class (1 = above threshold): 1


**Function 4: final query selection**

The best candidate was produced by the random search rather than by the neural-network gradient candidates. Using the selected A_original Gaussian Process, the chosen point is 0.367088-0.474599-0.423453-0.415817.

Its GP predicted mean is approximately -0.3905, with a relatively large predictive standard deviation of approximately 0.4679. The resulting UCB score of approximately 0.5454 therefore contains a substantial uncertainty contribution, making this proposal partly exploratory rather than purely exploitative.

The candidate lies approximately 0.0999 away from the nearest existing observation, so it explores a region that is not identical to the previously sampled points while remaining reasonably close to observed data. The SVM assigns class 1, indicating that the classifier places it above the classification threshold, although this does not guarantee an improvement in the numerical objective.

Because the random-search candidate achieved a higher GP-UCB score than the supplementary neural-network gradient candidates, I retained the random candidate for Function 4, round 4.


In [27]:
# Use all Function 5 observations available for round 4
X_5 = updated_data[5]["X"]
y_5 = updated_data[5]["y"]

models_5 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_5.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_5, y_5)
    models_5[name] = model

    print(f"\nModel: {name}")
    print("Observations:", len(y_5))
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )


Model: A_original
Observations: 23
Fitted kernel: 1.85**2 * Matern(length_scale=[0.631, 1.93, 0.815, 0.994], nu=2.5) + WhiteKernel(noise_level=7.41e-10)
Log-marginal likelihood: -21.200383344137848

Model: B_wider
Observations: 23
Fitted kernel: 1.4**2 * Matern(length_scale=[0.484, 10, 0.752, 0.481], nu=2.5) + WhiteKernel(noise_level=5.5e-10)
Log-marginal likelihood: -20.33859945510904


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


**Function 5: Gaussian Process model comparison**

For Function 5, the wider-bound Gaussian Process provides a better fit to the observed data than the original model. The log-marginal likelihood improves from approximately -21.2004 for A_original to -20.3386 for B_wider.

The fitted length scales also change noticeably between the two models. In particular, the second dimension reaches a length scale of 10.0 in B_wider, which is exactly the upper optimisation bound. Scikit-learn therefore issues a convergence warning indicating that the optimum may lie beyond the currently allowed range.

A large length scale means that the Gaussian Process considers the objective to vary relatively slowly along that dimension. Because the optimum is constrained by the current bound, selecting B_wider immediately could artificially restrict the model.

For this reason, I will test an additional model with a larger upper bound before selecting the Gaussian Process used for the Function 5 acquisition search.


In [28]:
# Test whether Function 5 still benefits from a wider length-scale bound
kernel_5_extra = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=[0.2] * X_5.shape[1],
        length_scale_bounds=(0.02, 50.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(1e-10, 1e-1)
    )
)

model_5_extra = GaussianProcessRegressor(
    kernel=kernel_5_extra,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

model_5_extra.fit(X_5, y_5)

models_5["C_extra_wide"] = model_5_extra

print("\nModel: C_extra_wide")
print("Observations:", len(y_5))
print("Fitted kernel:", model_5_extra.kernel_)
print(
    "Log-marginal likelihood:",
    model_5_extra.log_marginal_likelihood_value_
)


Model: C_extra_wide
Observations: 23
Fitted kernel: 1.46**2 * Matern(length_scale=[0.5, 50, 0.78, 0.494], nu=2.5) + WhiteKernel(noise_level=7.07e-09)
Log-marginal likelihood: -20.20198182661916


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 50.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


**Function 5: additional length-scale bound check**

The additional C_extra_wide model slightly improves the log-marginal likelihood compared with B_wider, from approximately -20.3386 to -20.2020.

However, the second length scale again reaches the newly increased upper bound, this time at 50.0. This indicates that the Gaussian Process continues to prefer an extremely smooth response along the second input dimension.

Although an even larger bound could be tested, the improvement in log-marginal likelihood is relatively small compared with the increase in the permitted length-scale range. Continuing to enlarge the bound would therefore add complexity without clear evidence that it would materially improve the acquisition search.

For this reason, I will stop increasing the bound here and retain the wider models as sensitivity checks when evaluating the next Function 5 candidate.


In [29]:
# Generate the same four-dimensional candidate set for all Function 5 GP models
rng_5 = np.random.default_rng(42)
random_candidates_5 = rng_5.uniform(
    0, 0.999999, size=(100000, X_5.shape[1])
)

# Round to portal-ready coordinates
random_candidates_5 = np.unique(
    np.round(random_candidates_5, 6),
    axis=0
)

# Include the candidates previously generated using NN gradients
nn_candidates_5 = np.asarray(gradient_candidates[5])

# Combine random and NN-gradient candidates
candidates_5 = np.unique(
    np.vstack([
        random_candidates_5,
        nn_candidates_5
    ]),
    axis=0
)

# Identify which candidates came from the NN-gradient experiment
is_nn_candidate_5 = np.any(
    np.all(
        candidates_5[:, None, :] == nn_candidates_5[None, :, :],
        axis=2
    ),
    axis=1
)

# Calculate distance to the nearest existing observation
nearest_distance_5 = np.min(
    np.linalg.norm(
        candidates_5[:, None, :] - X_5[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude previously evaluated points
keep_5 = nearest_distance_5 > 1e-6

candidates_5 = candidates_5[keep_5]
is_nn_candidate_5 = is_nn_candidate_5[keep_5]
nearest_distance_5 = nearest_distance_5[keep_5]

# Compare all GP models using the same UCB criterion
beta_5 = 2.0
proposals_5 = {}

for name, model in models_5.items():

    mean, std = model.predict(
        candidates_5,
        return_std=True
    )

    ucb = mean + beta_5 * std

    best_index = int(np.argmax(ucb))
    query = candidates_5[best_index].copy()

    svm_class = int(
        selected_boundary_svms[5].predict(
            query.reshape(1, -1)
        )[0]
    )

    proposals_5[name] = {
        "query": query,
        "mean": float(mean[best_index]),
        "std": float(std[best_index]),
        "ucb": float(ucb[best_index]),
        "nearest_distance": float(
            nearest_distance_5[best_index]
        ),
        "svm_class": svm_class,
        "candidate_source": (
            "NN gradient"
            if is_nn_candidate_5[best_index]
            else "Random search"
        )
    }

    print(f"\nModel: {name}")
    print(
        "Proposed query:",
        "-".join(f"{x:.6f}" for x in query)
    )
    print(
        "Candidate source:",
        proposals_5[name]["candidate_source"]
    )
    print(
        f"Predicted output: "
        f"{mean[best_index]:.6e}"
    )
    print(
        f"Predictive standard deviation: "
        f"{std[best_index]:.6e}"
    )
    print(
        f"UCB score: "
        f"{ucb[best_index]:.6e}"
    )
    print(
        f"Distance to nearest observation: "
        f"{nearest_distance_5[best_index]:.6f}"
    )
    print(
        "SVM class (1 = above threshold):",
        svm_class
    )


Model: A_original
Proposed query: 0.706599-0.999023-0.993654-0.986455
Candidate source: Random search
Predicted output: 4.350279e+03
Predictive standard deviation: 4.332983e+02
UCB score: 5.216876e+03
Distance to nearest observation: 0.202625
SVM class (1 = above threshold): 1

Model: B_wider
Proposed query: 0.622329-0.317899-0.999131-0.997718
Candidate source: Random search
Predicted output: 4.305302e+03
Predictive standard deviation: 2.760754e+02
UCB score: 4.857453e+03
Distance to nearest observation: 0.467617
SVM class (1 = above threshold): 1

Model: C_extra_wide
Proposed query: 0.622329-0.317899-0.999131-0.997718
Candidate source: Random search
Predicted output: 4.374767e+03
Predictive standard deviation: 2.501756e+02
UCB score: 4.875118e+03
Distance to nearest observation: 0.467617
SVM class (1 = above threshold): 1


**Function 5: candidate comparison and model selection**

The three Gaussian Process models were evaluated over the same candidate set, which combined 100,000 random-search points with the supplementary neural-network gradient candidates.

A_original proposes a different candidate and obtains the largest model-specific UCB score. However, this model also had the lowest log-marginal likelihood in the previous comparison, indicating a poorer fit to the observed Function 5 data.

Both B_wider and C_extra_wide independently select exactly the same candidate, 0.622329-0.317899-0.999131-0.997718. This is particularly useful because C_extra_wide allowed a much larger length-scale bound than B_wider, yet the acquisition optimum remained unchanged. The selected location is therefore robust to the additional length-scale sensitivity test.

Although C_extra_wide achieved a slightly higher log-marginal likelihood than B_wider, its second length scale again reached the imposed upper bound of 50.0. The improvement in model fit was relatively small, while B_wider already represented a substantial relaxation of the original bounds and produced the same acquisition candidate.

For this reason, I retain B_wider as the main Function 5 Gaussian Process and treat C_extra_wide as a sensitivity check confirming the stability of the proposed query.

The selected candidate comes from the random search rather than the neural-network gradient candidates. It has a predicted output of approximately 4305.30, predictive standard deviation of approximately 276.08, and UCB score of approximately 4857.45. Its distance to the nearest existing observation is approximately 0.468, indicating substantial exploration of a less densely sampled region. The SVM also assigns the candidate to class 1.


In [30]:
# Keep B_wider as the main Function 5 model
# C_extra_wide acts as a sensitivity check and proposes the same query
selected_model_5 = models_5["B_wider"]

# Store the selected round-4 proposal
query_round4_5 = proposals_5["B_wider"]["query"].copy()
proposal_round4_5 = proposals_5["B_wider"].copy()

print(
    "Function 5 — round 4:",
    "-".join(f"{x:.6f}" for x in query_round4_5)
)
print("Selected model: B_wider")
print(
    "Candidate source:",
    proposal_round4_5["candidate_source"]
)
print(
    f"Predicted output: "
    f"{proposal_round4_5['mean']:.6e}"
)
print(
    f"Predictive standard deviation: "
    f"{proposal_round4_5['std']:.6e}"
)
print(
    f"UCB score: "
    f"{proposal_round4_5['ucb']:.6e}"
)
print(
    f"Distance to nearest observation: "
    f"{proposal_round4_5['nearest_distance']:.6f}"
)
print(
    "SVM class (1 = above threshold):",
    proposal_round4_5["svm_class"]
)

Function 5 — round 4: 0.622329-0.317899-0.999131-0.997718
Selected model: B_wider
Candidate source: Random search
Predicted output: 4.305302e+03
Predictive standard deviation: 2.760754e+02
UCB score: 4.857453e+03
Distance to nearest observation: 0.467617
SVM class (1 = above threshold): 1


**Function 5: stored round-4 selection**

The final Function 5 round-4 query is 0.622329-0.317899-0.999131-0.997718.

B_wider is retained as the main Gaussian Process because it provides a better fit than A_original while avoiding the unnecessarily extreme upper bound used in C_extra_wide. Importantly, both wider models independently selected exactly the same acquisition candidate, which supports the stability of this proposal.

The selected point comes from the random-search candidate set. Its predicted output is approximately 4305.30, with a predictive standard deviation of approximately 276.08 and a UCB score of approximately 4857.45.

The point is approximately 0.468 away from the nearest existing observation, so it introduces a meaningful exploratory component while still being selected as promising by the Gaussian Process. The SVM assigns it to class 1, providing an additional indication that the candidate lies in the favourable region identified by the classifier.

This query is therefore retained as the Function 5 round-4 submission candidate.


In [31]:
# Use all Function 6 observations available for round 4
X_6 = updated_data[6]["X"]
y_6 = updated_data[6]["y"]

models_6 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_6.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_6, y_6)
    models_6[name] = model

    print(f"\nModel: {name}")
    print("Observations:", len(y_6))
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )


Model: A_original
Observations: 23
Fitted kernel: 1.44**2 * Matern(length_scale=[0.604, 1.1, 1.16, 1.19, 1.3], nu=2.5) + WhiteKernel(noise_level=0.024)
Log-marginal likelihood: -23.394751767745042

Model: B_wider
Observations: 23
Fitted kernel: 1.44**2 * Matern(length_scale=[0.604, 1.1, 1.16, 1.19, 1.3], nu=2.5) + WhiteKernel(noise_level=0.024)
Log-marginal likelihood: -23.394751771270126


**Function 6: Gaussian Process model comparison**

The A_original and B_wider Gaussian Process models converge to effectively identical solutions.

Both models produce the same fitted kernel, with length scales of approximately 0.604, 1.10, 1.16, 1.19 and 1.30, and the same estimated noise level of approximately 0.024.

Their log-marginal likelihood values are also virtually identical, differing only at numerical precision level. Importantly, none of the fitted length scales reaches the original upper bound of 2.0.

This indicates that the original length-scale bounds are not constraining the Function 6 Gaussian Process. Increasing the upper bound to 10.0 therefore provides no meaningful improvement in model fit.

For this reason, I retain A_original for Function 6 and proceed with the acquisition search using the original Gaussian Process. The candidate set will include both random-search points and the supplementary neural-network gradient candidates generated earlier.


In [32]:
selected_model_6 = models_6["A_original"]

In [33]:
# Generate five-dimensional candidate points for Function 6
rng_6 = np.random.default_rng(42)

random_candidates_6 = rng_6.uniform(
    0, 0.999999,
    size=(100000, X_6.shape[1])
)

# Round to portal-ready coordinates
random_candidates_6 = np.unique(
    np.round(random_candidates_6, 6),
    axis=0
)

# Include the supplementary NN-gradient candidates
nn_candidates_6 = np.asarray(
    gradient_candidates[6]
)

# Combine random and NN-gradient candidates
candidates_6 = np.unique(
    np.vstack([
        random_candidates_6,
        nn_candidates_6
    ]),
    axis=0
)

# Identify which candidates came from the NN-gradient experiment
is_nn_candidate_6 = np.any(
    np.all(
        candidates_6[:, None, :] ==
        nn_candidates_6[None, :, :],
        axis=2
    ),
    axis=1
)

# Calculate distance to the nearest existing observation
nearest_distance_6 = np.min(
    np.linalg.norm(
        candidates_6[:, None, :] -
        X_6[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude previously evaluated points
keep_6 = nearest_distance_6 > 1e-6

candidates_6 = candidates_6[keep_6]
is_nn_candidate_6 = is_nn_candidate_6[keep_6]
nearest_distance_6 = nearest_distance_6[keep_6]

# Evaluate the selected Gaussian Process
mean_6, std_6 = selected_model_6.predict(
    candidates_6,
    return_std=True
)

beta_6 = 2.0
ucb_6 = mean_6 + beta_6 * std_6

best_index_6 = int(
    np.argmax(ucb_6)
)

query_round4_6 = candidates_6[
    best_index_6
].copy()

svm_class_6 = int(
    selected_boundary_svms[6].predict(
        query_round4_6.reshape(1, -1)
    )[0]
)

candidate_source_6 = (
    "NN gradient"
    if is_nn_candidate_6[best_index_6]
    else "Random search"
)

print(
    "Function 6 — round 4:",
    "-".join(
        f"{x:.6f}"
        for x in query_round4_6
    )
)

print(
    "Selected model: A_original"
)

print(
    "Candidate source:",
    candidate_source_6
)

print(
    f"Predicted output: "
    f"{mean_6[best_index_6]:.6e}"
)

print(
    f"Predictive standard deviation: "
    f"{std_6[best_index_6]:.6e}"
)

print(
    f"UCB score: "
    f"{ucb_6[best_index_6]:.6e}"
)

print(
    f"Distance to nearest observation: "
    f"{nearest_distance_6[best_index_6]:.6f}"
)

print(
    "SVM class (1 = above threshold):",
    svm_class_6
)

Function 6 — round 4: 0.551327-0.429440-0.664216-0.837319-0.028275
Selected model: A_original
Candidate source: Random search
Predicted output: -6.067783e-01
Predictive standard deviation: 1.556102e-01
UCB score: -2.955579e-01
Distance to nearest observation: 0.338641
SVM class (1 = above threshold): 1


**Function 6: round-4 candidate selection**

The selected Function 6 round-4 query is 0.551327-0.429440-0.664216-0.837319-0.028275.

The candidate was selected using the A_original Gaussian Process, since the previous model comparison showed that A_original and B_wider converged to effectively identical fitted kernels and log-marginal likelihoods. The wider bounds therefore provided no meaningful benefit for Function 6.

The acquisition search combined 100,000 random-search candidates with the supplementary neural-network gradient candidates. The final point was selected from the random-search set, indicating that none of the NN-gradient candidates achieved a higher GP-UCB value.

The Gaussian Process predicts an output of approximately -0.607, with a predictive standard deviation of approximately 0.156. Using the exploration parameter β = 2.0, this gives a UCB score of approximately -0.296.

Although the predicted mean remains negative, the acquisition function considers both the expected value and predictive uncertainty. The selected candidate is approximately 0.339 away from the nearest existing observation, meaning that it explores a reasonably distinct region of the five-dimensional search space.

The SVM assigns the candidate to class 1, indicating that it lies in the region classified as being above the threshold used in the supplementary classification analysis.

This query is therefore retained as the Function 6 round-4 submission candidate.


In [35]:
# Store the complete Function 6 round-4 proposal
proposal_round4_6 = {
    "query": query_round4_6.copy(),
    "model": "A_original",
    "candidate_source": candidate_source_6,
    "mean": float(mean_6[best_index_6]),
    "std": float(std_6[best_index_6]),
    "ucb": float(ucb_6[best_index_6]),
    "nearest_distance": float(
        nearest_distance_6[best_index_6]
    ),
    "svm_class": svm_class_6
}

print("Function 6 proposal stored successfully.")
print(proposal_round4_6)

Function 6 proposal stored successfully.
{'query': array([0.551327, 0.42944 , 0.664216, 0.837319, 0.028275]), 'model': 'A_original', 'candidate_source': 'Random search', 'mean': -0.6067782964233326, 'std': 0.15561022039281158, 'ucb': -0.2955578556377094, 'nearest_distance': 0.33864088251075647, 'svm_class': 1}


**Function 6: proposal stored**

The complete Function 6 round-4 proposal has now been stored in `proposal_round4_6`.

This dictionary preserves the selected query together with the GP model, candidate source, predicted mean, predictive uncertainty, UCB score, distance to the nearest observation and SVM classification.

Keeping these values together will make it easier to compare all eight Function 1–8 round-4 proposals at the end of the notebook.


In [36]:
# Use all Function 7 observations available for round 4
X_7 = updated_data[7]["X"]
y_7 = updated_data[7]["y"]

models_7 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_7.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_7, y_7)
    models_7[name] = model

    print(f"\nModel: {name}")
    print("Observations:", len(y_7))
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )

c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



Model: A_original
Observations: 33
Fitted kernel: 0.901**2 * Matern(length_scale=[1.07, 2, 2, 0.337, 0.167, 0.101], nu=2.5) + WhiteKernel(noise_level=3.43e-09)
Log-marginal likelihood: -38.906895215918276

Model: B_wider
Observations: 33
Fitted kernel: 1.05**2 * Matern(length_scale=[1.03, 10, 0.486, 0.514, 0.277, 0.514], nu=2.5) + WhiteKernel(noise_level=2.54e-06)
Log-marginal likelihood: -37.90999048571518


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


**Function 7: Gaussian Process model comparison**

The wider-bound Gaussian Process provides a better fit to the Function 7 observations than the original model. The log-marginal likelihood improves from approximately -38.9069 for A_original to -37.9100 for B_wider.

In A_original, the second and third length scales reach the original upper bound of 2.0, indicating that the model is constrained in those dimensions.

After increasing the upper bound to 10.0, the fitted kernel changes substantially. The second length scale increases to 10.0 and again reaches the new upper bound, while the remaining dimensions settle at finite values below the limit.

This indicates that Function 7 appears to vary very slowly along the second input dimension and that the GP would prefer an even larger length scale there.

Because the wider model improves the log-marginal likelihood but still reaches its upper bound, I will perform one additional sensitivity check with a larger length-scale limit before selecting the Gaussian Process used for the Function 7 acquisition search.


In [37]:
# Test whether Function 7 still benefits from a wider length-scale bound
kernel_7_extra = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=[0.2] * X_7.shape[1],
        length_scale_bounds=(0.02, 50.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(1e-10, 1e-1)
    )
)

model_7_extra = GaussianProcessRegressor(
    kernel=kernel_7_extra,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

model_7_extra.fit(X_7, y_7)

models_7["C_extra_wide"] = model_7_extra

print("\nModel: C_extra_wide")
print("Observations:", len(y_7))
print("Fitted kernel:", model_7_extra.kernel_)
print(
    "Log-marginal likelihood:",
    model_7_extra.log_marginal_likelihood_value_
)


Model: C_extra_wide
Observations: 33
Fitted kernel: 1.07**2 * Matern(length_scale=[50, 50, 1.32, 0.377, 50, 0.0353], nu=2.5) + WhiteKernel(noise_level=1.94e-09)
Log-marginal likelihood: -38.14507459271026


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 50.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 50.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 50.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


**Function 7: additional length-scale bound check**

The additional C_extra_wide model does not improve the Function 7 Gaussian Process fit relative to B_wider.

Its log-marginal likelihood is approximately -38.1451, which is better than A_original (-38.9069) but worse than B_wider (-37.9100).

The fitted kernel also becomes substantially more extreme. Three dimensions reach the enlarged upper bound of 50.0, indicating that the optimisation is treating several directions as nearly flat across the currently observed region.

Because increasing the length-scale bound from 10.0 to 50.0 does not improve the log-marginal likelihood and produces a more extreme kernel, there is no evidence that continuing to increase the bound would provide a better model for the acquisition search.

For this reason, B_wider is retained as the preferred Gaussian Process for Function 7.


In [38]:
# Retain the best-supported Gaussian Process for Function 7
selected_model_7 = models_7["B_wider"]

**Function 7: selected Gaussian Process**

B_wider is selected for the Function 7 acquisition search because it achieves the highest log-marginal likelihood among the three tested models without requiring the much more extreme length scales observed in C_extra_wide.


In [39]:
# Generate six-dimensional random candidates for Function 7
rng_7 = np.random.default_rng(42)

candidates_7 = rng_7.uniform(
    0, 0.999999,
    size=(100000, X_7.shape[1])
)

# Round to portal-ready coordinates
candidates_7 = np.unique(
    np.round(candidates_7, 6),
    axis=0
)

# Calculate distance to the nearest existing observation
nearest_distance_7 = np.min(
    np.linalg.norm(
        candidates_7[:, None, :] -
        X_7[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude previously evaluated points
keep_7 = nearest_distance_7 > 1e-6

candidates_7 = candidates_7[keep_7]
nearest_distance_7 = nearest_distance_7[keep_7]

# Evaluate the selected Gaussian Process
mean_7, std_7 = selected_model_7.predict(
    candidates_7,
    return_std=True
)

beta_7 = 2.0
ucb_7 = mean_7 + beta_7 * std_7

best_index_7 = int(
    np.argmax(ucb_7)
)

query_round4_7 = candidates_7[
    best_index_7
].copy()

svm_class_7 = int(
    selected_boundary_svms[7].predict(
        query_round4_7.reshape(1, -1)
    )[0]
)

candidate_source_7 = "Random search"

print(
    "Function 7 — round 4:",
    "-".join(
        f"{x:.6f}"
        for x in query_round4_7
    )
)

print(
    "Selected model: B_wider"
)

print(
    "Candidate source:",
    candidate_source_7
)

print(
    f"Predicted output: "
    f"{mean_7[best_index_7]:.6e}"
)

print(
    f"Predictive standard deviation: "
    f"{std_7[best_index_7]:.6e}"
)

print(
    f"UCB score: "
    f"{ucb_7[best_index_7]:.6e}"
)

print(
    f"Distance to nearest observation: "
    f"{nearest_distance_7[best_index_7]:.6f}"
)

print(
    "SVM class (1 = above threshold):",
    svm_class_7
)

Function 7 — round 4: 0.048559-0.653060-0.672739-0.057497-0.335520-0.902837
Selected model: B_wider
Candidate source: Random search
Predicted output: 1.886739e+00
Predictive standard deviation: 2.020933e-01
UCB score: 2.290926e+00
Distance to nearest observation: 0.439762
SVM class (1 = above threshold): 1


**Function 7: round-4 candidate selection**

The selected Function 7 round-4 query is 0.048559-0.653060-0.672739-0.057497-0.335520-0.902837.

The candidate was selected using the B_wider Gaussian Process. This model was retained because it achieved the highest log-marginal likelihood among the tested alternatives, while the more extreme C_extra_wide sensitivity model did not improve the fit.

For Function 7, the acquisition search used 100,000 random-search candidates. Neural-network gradient candidates were not included for this function because the previous neural-network analysis did not provide sufficiently reliable regression behaviour to justify using them as supplementary acquisition candidates.

The selected point has a predicted output of approximately 1.887 and a predictive standard deviation of approximately 0.202. With β = 2.0, this produces a UCB score of approximately 2.291.

Its distance to the nearest existing observation is approximately 0.440, meaning that the proposal explores a relatively distinct part of the six-dimensional search space rather than concentrating very close to an already evaluated point.

The SVM assigns the candidate to class 1, providing an additional indication that the point lies within the region classified as favourable by the supplementary boundary model.

This query is therefore retained as the Function 7 round-4 submission candidate.


In [40]:
# Use all Function 8 observations available for round 4
X_8 = updated_data[8]["X"]
y_8 = updated_data[8]["y"]

models_8 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_8.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_8, y_8)
    models_8[name] = model

    print(f"\nModel: {name}")
    print("Observations:", len(y_8))
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )

c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-pack


Model: A_original
Observations: 43
Fitted kernel: 0.991**2 * Matern(length_scale=[1.18, 2, 0.914, 2, 2, 2, 1.27, 2], nu=2.5) + WhiteKernel(noise_level=1.1e-10)
Log-marginal likelihood: -13.820054926463968

Model: B_wider
Observations: 43
Fitted kernel: 3.14**2 * Matern(length_scale=[3.03, 4.28, 2.21, 7.97, 10, 4.39, 2.97, 10], nu=2.5) + WhiteKernel(noise_level=1e-10)
Log-marginal likelihood: 1.284772211652374


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


**Function 8: Gaussian Process model comparison**

The wider-bound Gaussian Process provides a substantially better fit to the Function 8 observations than the original model.

The log-marginal likelihood improves from approximately -13.8201 for A_original to approximately 1.2848 for B_wider, representing a very large improvement in model fit.

In A_original, five of the eight fitted length scales reach the original upper bound of 2.0. This indicates that the original Gaussian Process is strongly constrained in several input dimensions.

After increasing the upper bound to 10.0, the fitted length scales expand considerably and the overall model fit improves dramatically. However, the fifth and eighth dimensions still reach the new upper bound of 10.0, suggesting that the model may prefer even smoother variation along those directions.

The estimated noise level is also driven to approximately the lower bound, indicating that the observed Function 8 data are being modelled as almost noise-free.

Because the wider model is clearly superior but still reaches the length-scale limit in two dimensions, I will perform one additional sensitivity check with a larger upper bound before selecting the Gaussian Process used for the Function 8 acquisition search.


In [41]:
# Test whether Function 8 still benefits from a wider length-scale bound
kernel_8_extra = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=[0.2] * X_8.shape[1],
        length_scale_bounds=(0.02, 50.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(1e-10, 1e-1)
    )
)

model_8_extra = GaussianProcessRegressor(
    kernel=kernel_8_extra,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

model_8_extra.fit(X_8, y_8)

models_8["C_extra_wide"] = model_8_extra

print("\nModel: C_extra_wide")
print("Observations:", len(y_8))
print("Fitted kernel:", model_8_extra.kernel_)
print(
    "Log-marginal likelihood:",
    model_8_extra.log_marginal_likelihood_value_
)


Model: C_extra_wide
Observations: 43
Fitted kernel: 5.53**2 * Matern(length_scale=[4.15, 6.14, 3.26, 7.16, 20.7, 6.06, 4.21, 50], nu=2.5) + WhiteKernel(noise_level=1e-10)
Log-marginal likelihood: 4.677426225934127


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__k2__length_scale is close to the specified upper bound 50.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


**Function 8: additional length-scale bound check**

The additional C_extra_wide Gaussian Process provides a further substantial improvement over B_wider.

The log-marginal likelihood increases from approximately 1.2848 for B_wider to approximately 4.6774 for C_extra_wide. This confirms that the upper length-scale bound of 10.0 was still constraining the Function 8 model.

With the wider bound, most dimensions settle at finite length scales below 50.0. Only the eighth dimension reaches the new upper bound, while the fifth dimension, which previously reached 10.0, now settles at approximately 20.7.

This suggests that the Gaussian Process benefits meaningfully from allowing smoother behaviour across several dimensions, particularly the eighth input dimension.

The estimated noise level remains at approximately the lower bound, so the model continues to treat the observed Function 8 responses as nearly noise-free.

Because C_extra_wide achieves the highest log-marginal likelihood by a clear margin and produces a less broadly constrained kernel than the previous models, I retain C_extra_wide as the preferred Gaussian Process for the Function 8 acquisition search.



In [43]:
# Retain the best-supported Gaussian Process for Function 8
selected_model_8 = models_8["C_extra_wide"]

**Function 8: selected Gaussian Process**

C_extra_wide is selected for the Function 8 acquisition search because it achieves the highest log-marginal likelihood of the tested models and substantially improves the fit relative to both A_original and B_wider.


In [44]:
# Generate eight-dimensional candidate points for Function 8
rng_8 = np.random.default_rng(42)

random_candidates_8 = rng_8.uniform(
    0, 0.999999,
    size=(100000, X_8.shape[1])
)

# Round to portal-ready coordinates
random_candidates_8 = np.unique(
    np.round(random_candidates_8, 6),
    axis=0
)

# Include the supplementary NN-gradient candidates
nn_candidates_8 = np.asarray(
    gradient_candidates[8]
)

# Combine random and NN-gradient candidates
candidates_8 = np.unique(
    np.vstack([
        random_candidates_8,
        nn_candidates_8
    ]),
    axis=0
)

# Identify which candidates came from the NN-gradient experiment
is_nn_candidate_8 = np.any(
    np.all(
        candidates_8[:, None, :] ==
        nn_candidates_8[None, :, :],
        axis=2
    ),
    axis=1
)

# Calculate distance to the nearest existing observation
nearest_distance_8 = np.min(
    np.linalg.norm(
        candidates_8[:, None, :] -
        X_8[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude previously evaluated points
keep_8 = nearest_distance_8 > 1e-6

candidates_8 = candidates_8[keep_8]
is_nn_candidate_8 = is_nn_candidate_8[keep_8]
nearest_distance_8 = nearest_distance_8[keep_8]

# Evaluate the selected Gaussian Process
mean_8, std_8 = selected_model_8.predict(
    candidates_8,
    return_std=True
)

beta_8 = 2.0
ucb_8 = mean_8 + beta_8 * std_8

best_index_8 = int(
    np.argmax(ucb_8)
)

query_round4_8 = candidates_8[
    best_index_8
].copy()

svm_class_8 = int(
    selected_boundary_svms[8].predict(
        query_round4_8.reshape(1, -1)
    )[0]
)

candidate_source_8 = (
    "NN gradient"
    if is_nn_candidate_8[best_index_8]
    else "Random search"
)

print(
    "Function 8 — round 4:",
    "-".join(
        f"{x:.6f}"
        for x in query_round4_8
    )
)

print(
    "Selected model: C_extra_wide"
)

print(
    "Candidate source:",
    candidate_source_8
)

print(
    f"Predicted output: "
    f"{mean_8[best_index_8]:.6e}"
)

print(
    f"Predictive standard deviation: "
    f"{std_8[best_index_8]:.6e}"
)

print(
    f"UCB score: "
    f"{ucb_8[best_index_8]:.6e}"
)

print(
    f"Distance to nearest observation: "
    f"{nearest_distance_8[best_index_8]:.6f}"
)

print(
    "SVM class (1 = above threshold):",
    svm_class_8
)

Function 8 — round 4: 0.235512-0.179280-0.060095-0.061301-0.940810-0.603515-0.198305-0.781321
Selected model: C_extra_wide
Candidate source: Random search
Predicted output: 9.901145e+00
Predictive standard deviation: 7.270987e-02
UCB score: 1.004656e+01
Distance to nearest observation: 0.355823
SVM class (1 = above threshold): 1


**Function 8: round-4 candidate selection**

The selected Function 8 round-4 query is 0.235512-0.179280-0.060095-0.061301-0.940810-0.603515-0.198305-0.781321.

The candidate was selected using the C_extra_wide Gaussian Process, which achieved the highest log-marginal likelihood among the tested Function 8 models and substantially improved the fit relative to both A_original and B_wider.

The acquisition search combined 100,000 random-search candidates with the supplementary neural-network gradient candidates. The final candidate was selected from the random-search set, meaning that none of the NN-gradient candidates achieved a higher GP-UCB score.

The Gaussian Process predicts an output of approximately 9.901, with a relatively small predictive standard deviation of approximately 0.073. With β = 2.0, this results in a UCB score of approximately 10.047.

The candidate is approximately 0.356 away from the nearest existing observation, providing some exploration of a new region while still being strongly supported by the Gaussian Process predicted mean.

The SVM assigns the candidate to class 1, providing an additional indication that it lies within the favourable region identified by the supplementary classification model.

This query is therefore retained as the Function 8 round-4 submission candidate.


In [45]:
# Standardise and store the Function 5–8 round-4 selections
# using the same naming convention already used for Functions 1–4

# Function 5
query_5_round4 = query_round4_5.copy()
selected_proposal_5_round4 = proposal_round4_5.copy()

# Function 6
query_6_round4 = query_round4_6.copy()

selected_proposal_6_round4 = {
    "query": query_6_round4.copy(),
    "mean": float(mean_6[best_index_6]),
    "std": float(std_6[best_index_6]),
    "ucb": float(ucb_6[best_index_6]),
    "nearest_distance": float(
        nearest_distance_6[best_index_6]
    ),
    "svm_class": int(svm_class_6),
    "candidate_source": candidate_source_6
}

# Function 7
query_7_round4 = query_round4_7.copy()

selected_proposal_7_round4 = {
    "query": query_7_round4.copy(),
    "mean": float(mean_7[best_index_7]),
    "std": float(std_7[best_index_7]),
    "ucb": float(ucb_7[best_index_7]),
    "nearest_distance": float(
        nearest_distance_7[best_index_7]
    ),
    "svm_class": int(svm_class_7),
    "candidate_source": candidate_source_7
}

# Function 8
query_8_round4 = query_round4_8.copy()

selected_proposal_8_round4 = {
    "query": query_8_round4.copy(),
    "mean": float(mean_8[best_index_8]),
    "std": float(std_8[best_index_8]),
    "ucb": float(ucb_8[best_index_8]),
    "nearest_distance": float(
        nearest_distance_8[best_index_8]
    ),
    "svm_class": int(svm_class_8),
    "candidate_source": candidate_source_8
}

print("Functions 5–8 round-4 selections stored successfully.")

Functions 5–8 round-4 selections stored successfully.


**Round 4: proposal storage**

The Function 5–8 round-4 selections have now been stored using the same naming convention already used for Functions 1–4.

For each function, `query_X_round4` contains the final portal-ready query, while `selected_proposal_X_round4` preserves the associated Gaussian Process prediction, predictive uncertainty, UCB score, distance to the nearest observation, SVM classification and candidate source.

All eight Function 1–8 round-4 selections can therefore now be handled consistently in the final summary and subsequent portal-feedback analysis.


In [47]:
# Standardise and store the Function 5–8 round-4 selections
# using the same naming convention already used for Functions 1–4

# Function 5
query_5_round4 = query_round4_5.copy()
selected_proposal_5_round4 = proposal_round4_5.copy()

# Function 6
query_6_round4 = query_round4_6.copy()

selected_proposal_6_round4 = {
    "query": query_6_round4.copy(),
    "mean": float(mean_6[best_index_6]),
    "std": float(std_6[best_index_6]),
    "ucb": float(ucb_6[best_index_6]),
    "nearest_distance": float(nearest_distance_6[best_index_6]),
    "svm_class": int(svm_class_6),
    "candidate_source": candidate_source_6
}

# Function 7
query_7_round4 = query_round4_7.copy()

selected_proposal_7_round4 = {
    "query": query_7_round4.copy(),
    "mean": float(mean_7[best_index_7]),
    "std": float(std_7[best_index_7]),
    "ucb": float(ucb_7[best_index_7]),
    "nearest_distance": float(nearest_distance_7[best_index_7]),
    "svm_class": int(svm_class_7),
    "candidate_source": candidate_source_7
}

# Function 8
query_8_round4 = query_round4_8.copy()

selected_proposal_8_round4 = {
    "query": query_8_round4.copy(),
    "mean": float(mean_8[best_index_8]),
    "std": float(std_8[best_index_8]),
    "ucb": float(ucb_8[best_index_8]),
    "nearest_distance": float(nearest_distance_8[best_index_8]),
    "svm_class": int(svm_class_8),
    "candidate_source": candidate_source_8
}

print("Functions 5–8 round-4 selections stored successfully.")

Functions 5–8 round-4 selections stored successfully.


**Round 4: proposal storage**

The Function 5–8 round-4 selections have now been stored using the same naming convention already used for Functions 1–4.

For each function, `query_X_round4` contains the final portal-ready query, while `selected_proposal_X_round4` preserves the associated Gaussian Process prediction, predictive uncertainty, UCB score, distance to the nearest observation, SVM classification and candidate source.

All eight Function 1–8 round-4 selections can therefore now be handled consistently in the final summary and subsequent portal-feedback analysis.


In [48]:
# Consolidate all Function 1–8 round-4 proposals

round4_proposals = {
    1: selected_proposal_1_round4,
    2: selected_proposal_2_round4,
    3: selected_proposal_3_round4,
    4: selected_proposal_4_round4,
    5: selected_proposal_5_round4,
    6: selected_proposal_6_round4,
    7: selected_proposal_7_round4,
    8: selected_proposal_8_round4,
}

round4_models = {
    1: "A_original",
    2: "B_wider",
    3: "B_wider",
    4: "A_original",
    5: "B_wider",
    6: "A_original",
    7: "B_wider",
    8: "C_extra_wide",
}

round4_rows = []

for function_id, proposal in round4_proposals.items():

    query = proposal["query"]

    round4_rows.append({
        "Function": function_id,
        "Query": "-".join(
            f"{x:.6f}" for x in query
        ),
        "Model": round4_models[function_id],
        "Candidate source": proposal["candidate_source"],
        "Predicted output": proposal["mean"],
        "Predictive std": proposal["std"],
        "UCB": proposal["ucb"],
        "Nearest distance": proposal["nearest_distance"],
        "SVM class": proposal["svm_class"],
    })

round4_summary = pd.DataFrame(round4_rows)

round4_summary

,Function,Query,Model,Candidate source,Predicted output,Predictive std,UCB,Nearest distance,SVM class
0,1,0.705334-0.019831,A_original,Random search,0.000010,0.000886,0.001782,0.207724,0
1,2,0.699810-0.002104,B_wider,Random search,0.663418,0.055003,0.773424,0.020943,1
2,3,0.933505-0.993849-0.852342,B_wider,Random search,-0.067064,0.078767,0.090470,0.316528,1
3,4,0.367088-0.474599-0.423453-0.415817,A_original,Random search,-0.390458,0.467940,0.545421,0.099941,1
4,5,0.622329-0.317899-0.999131-0.997718,B_wider,Random search,4305.301966,276.075382,4857.452731,0.467617,1
5,6,0.551327-0.429440-0.664216-0.837319-0.028275,A_original,Random search,-0.606778,0.155610,-0.295558,0.338641,1
6,7,0.048559-0.653060-0.672739-0.057497-0.335520-0...,B_wider,Random search,1.886739,0.202093,2.290926,0.439762,1
7,8,0.235512-0.179280-0.060095-0.061301-0.940810-0...,C_extra_wide,Random search,9.901145,0.072710,10.046565,0.355823,1


**Round 4: final proposal summary**

The final round-4 candidate queries for all eight black-box functions have now been consolidated into a single summary table.

Each proposal was selected using a Gaussian Process acquisition strategy based on the UCB criterion, with model selection performed separately for each function according to the fitted kernels, log-marginal likelihoods and length-scale sensitivity checks.

The selected Gaussian Process models differ across functions because the available observations support different levels of smoothness and complexity. Functions 1, 4 and 6 retain the original Gaussian Process bounds, Functions 2, 3, 5 and 7 use the wider-bound models, and Function 8 uses the extra-wide model after this produced a substantial improvement in log-marginal likelihood.

All final round-4 candidates originate from the random-search candidate sets. For Functions 4, 5, 6 and 8, supplementary neural-network gradient candidates were also included in the acquisition search, but none achieved a higher UCB score than the selected random-search proposal.

The selected candidates generally maintain a meaningful distance from the nearest existing observation, allowing the acquisition strategy to explore new regions of the search space rather than repeatedly sampling very close to existing points.

The supplementary SVM classification assigns class 1 to the selected candidates for Functions 2–8. Function 1 is the only selected proposal classified as class 0, so its selection is driven primarily by the Gaussian Process acquisition criterion rather than agreement with the supplementary classifier.

The UCB values should be interpreted within each individual function rather than compared directly across functions, since the black-box objectives operate on very different output scales.

These eight queries constitute the final round-4 submission candidates and can now be submitted to obtain the next set of black-box observations.


In [49]:
# Print all final round-4 queries in portal-ready format

print("FINAL ROUND-4 QUERIES")
print()

for function_id in range(1, 9):
    query = globals()[f"query_{function_id}_round4"]

    print(
        f"Function {function_id}: "
        + "-".join(f"{x:.6f}" for x in query)
    )

FINAL ROUND-4 QUERIES

Function 1: 0.705334-0.019831
Function 2: 0.699810-0.002104
Function 3: 0.933505-0.993849-0.852342
Function 4: 0.367088-0.474599-0.423453-0.415817
Function 5: 0.622329-0.317899-0.999131-0.997718
Function 6: 0.551327-0.429440-0.664216-0.837319-0.028275
Function 7: 0.048559-0.653060-0.672739-0.057497-0.335520-0.902837
Function 8: 0.235512-0.179280-0.060095-0.061301-0.940810-0.603515-0.198305-0.781321


**Round 4: portal-ready queries**

The final round-4 queries are printed below in the exact six-decimal format required for submission.

This final output is generated directly from the stored `query_X_round4` variables, reducing the risk of transcription errors when copying the candidates into the Black-Box Optimisation portal.
